# 07 — Audio and Speech Multimodal Processing

## What
Audio multimodal AI covers automatic speech recognition (ASR), text-to-speech (TTS), speaker diarisation, and audio understanding. The dominant architecture for ASR is the Whisper transformer encoder-decoder trained on 680k hours of labelled audio. Audio is first converted to log-Mel spectrograms, then processed as a 2D feature map.

## Why
Speech is the most natural human modality. Voice assistants, meeting transcription, medical dictation, and accessibility tools all depend on robust ASR. Audio understanding — classifying sounds, detecting emotion, identifying speakers — is increasingly relevant for multimodal systems that process video or real-world sensor streams.

## Community context
Whisper (OpenAI, 2022) set a new bar for zero-shot ASR by training on 680k hours of weakly supervised web audio. wav2vec 2.0 (Facebook) pioneered self-supervised audio pretraining — learning speech representations without labels. HuBERT extends this with masked prediction in feature space. SeamlessM4T (Meta, 2023) added translation across 100 languages.

In [ ]:
# Log-Mel spectrogram computation — the standard audio front-end
import numpy as np

def compute_log_mel_spectrogram(audio, sr=16000, n_fft=400, hop_length=160,
                                 n_mels=80, fmin=0, fmax=8000):
    """
    Compute log-Mel spectrogram from raw audio waveform.
    Standard front-end for Whisper: 80 Mel bins, 25ms windows, 10ms hop.
    """
    # STFT using numpy (simplified — real implementations use librosa/torchaudio)
    n_frames = (len(audio) - n_fft) // hop_length + 1
    stft_mag = np.zeros((n_fft//2+1, n_frames))
    window = np.hanning(n_fft)
    for i in range(n_frames):
        frame = audio[i*hop_length : i*hop_length+n_fft]
        if len(frame) < n_fft:
            frame = np.pad(frame, (0, n_fft-len(frame)))
        stft_mag[:, i] = np.abs(np.fft.rfft(frame * window))
    power = stft_mag ** 2
    # Build Mel filterbank
    freqs = np.fft.rfftfreq(n_fft, d=1/sr)
    mel_freqs = np.linspace(fmin, fmax, n_mels+2)
    filterbank = np.zeros((n_mels, len(freqs)))
    for m in range(n_mels):
        lo, center, hi = mel_freqs[m], mel_freqs[m+1], mel_freqs[m+2]
        for k, f in enumerate(freqs):
            if lo <= f <= center:
                filterbank[m, k] = (f-lo) / (center-lo+1e-8)
            elif center < f <= hi:
                filterbank[m, k] = (hi-f) / (hi-center+1e-8)
    mel_power = filterbank @ power  # (n_mels, n_frames)
    log_mel = np.log(mel_power + 1e-10)
    return log_mel

# Generate 1 second of synthetic audio (440 Hz tone + noise)
sr = 16000
t = np.linspace(0, 1, sr)
audio = 0.5 * np.sin(2*np.pi*440*t) + 0.1*np.random.randn(sr)

log_mel = compute_log_mel_spectrogram(audio, sr=sr)
print(f'Audio length: {len(audio)} samples ({len(audio)/sr:.1f}s at {sr}Hz)')
print(f'Log-Mel spectrogram shape: {log_mel.shape}  (n_mels x n_frames)')
print(f'  n_mels={log_mel.shape[0]}  n_frames={log_mel.shape[1]}')
print(f'  Each frame = 10ms hop -> {log_mel.shape[1]*10}ms coverage')
print(f'  Value range: [{log_mel.min():.2f}, {log_mel.max():.2f}]')
print('\nThis 2D spectrogram is the input to the Whisper encoder (like an image to ViT)')

## Whisper Architecture

Whisper uses a standard transformer encoder-decoder. The encoder processes 30-second log-Mel spectrogram segments (padded/trimmed to 30s × 80 Mel bins). The decoder autoregressively generates token IDs corresponding to transcribed text, language, and task tokens.

In [ ]:
# Whisper-style encoder: spectrogram -> contextual features
class AudioEncoder:
    """
    Simulates Whisper encoder:
    1. Two conv layers for local feature extraction
    2. Positional embedding (sinusoidal)
    3. Transformer blocks (here single-layer for demo)
    """
    def __init__(self, n_mels=80, embed_dim=64, n_frames_out=100):
        self.embed_dim = embed_dim
        # Simulate conv downsampling
        self.W_conv = np.random.randn(n_mels, embed_dim) * 0.01
        # Positional embedding
        self.pos_embed = self._sinusoidal_pos(n_frames_out, embed_dim)

    def _sinusoidal_pos(self, max_len, dim):
        pos = np.arange(max_len)[:, None]
        i = np.arange(dim//2)[None, :]
        angles = pos / 10000 ** (2*i/dim)
        pe = np.zeros((max_len, dim))
        pe[:, 0::2] = np.sin(angles)
        pe[:, 1::2] = np.cos(angles)
        return pe

    def __call__(self, log_mel):
        """log_mel: (n_mels, n_frames) -> features: (n_frames', embed_dim)"""
        # Simple projection: treat mel channels as feature dim
        x = log_mel.T @ self.W_conv   # (n_frames, embed_dim)
        # Subsample to n_frames_out
        n = min(len(x), len(self.pos_embed))
        x = x[:n]
        x = x + self.pos_embed[:n]
        return x

encoder = AudioEncoder(n_mels=80, embed_dim=64)
audio_features = encoder(log_mel)
print(f'Log-Mel input:       {log_mel.shape}')
print(f'Encoder output:      {audio_features.shape}  (frame embeddings)')
print(f'Each frame = 10ms -> {audio_features.shape[0]*10}ms audio encoded')
print('\nDecoder would autoregressively generate text tokens attending to these features')

# Simulate WER (Word Error Rate) — the key ASR metric
def wer(reference, hypothesis):
    r, h = reference.split(), hypothesis.split()
    d = np.zeros((len(r)+1, len(h)+1), dtype=int)
    for i in range(len(r)+1): d[i,0] = i
    for j in range(len(h)+1): d[0,j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i,j] = d[i-1,j-1] if r[i-1]==h[j-1] else 1+min(d[i-1,j],d[i,j-1],d[i-1,j-1])
    return d[len(r),len(h)] / max(len(r), 1)

ref = 'the quick brown fox jumps over the lazy dog'
hyp = 'the quick brown fox jumped over a lazy dog'
print(f'\nWER example: {wer(ref, hyp):.3f}  ({int(wer(ref,hyp)*9)}/9 words wrong)')